In [1]:
#데이터에서 누락된 값(NaN - 판다스에서는 기본적으로 NaN으로 표시)을 찾고 수정하거나 삭제하기
import gdown
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
import re

gdown.download(r'https://bit.ly/3GisL6J', 'ns_book4.csv', quiet=False)
ns_book4 = pd.read_csv('ns_book4.csv', low_memory=False)

Downloading...
From: https://bit.ly/3GisL6J
To: c:\data\ns_book4.csv
100%|██████████| 55.5M/55.5M [00:07<00:00, 7.34MB/s]


In [2]:
ns_book4 = ns_book4.astype({'도서권수':'int64', '대출건수':'int64'})
ns_book4.isna().sum()

set_ISBN_row = ns_book4['세트 ISBN'].isna()
ns_book4.loc[set_ISBN_row, '세트 ISBN'] = ''
ns_book4['세트 ISBN']

0                      
1                      
2                      
3                      
4                      
              ...      
384586    9788942300365
384587    9788974231309
384588    9788977352988
384589    9788985367424
384590    9788947700405
Name: 세트 ISBN, Length: 384591, dtype: object

In [3]:
#replace 정규식
invalid_number = ns_book4['발행년도'].str.contains(r'\D', na=True)
print(invalid_number.sum())
ns_book4[invalid_number]

ns_book5 = ns_book4.replace({'발행년도': {r'.*(\d{4}).*': r'\1'}}, regex=True)
numbers = ns_book5['발행년도'].str.contains(r'\D', na=True)
print(numbers.sum())



1777
67


In [4]:
ns_book5.loc[numbers, '발행년도'] = '-1'
ns_book5[numbers]

ns_book5 = ns_book5.astype({'발행년도':'int64'})

#지나치게 큰/작은 연도 값 찾고 수정한뒤 -1로 바꾸기
dk_year_row = ns_book5['발행년도'].gt(4000)
ns_book5.loc[dk_year_row, '발행년도'] = ns_book5.loc[dk_year_row, '발행년도'] - 2333




In [5]:
overyear_row = ns_book5['발행년도'].gt(4000)
ns_book5.loc[overyear_row, '발행년도'] = -1

lowyear_row = ns_book5['발행년도'].gt(0) & ns_book5['발행년도'].lt(1900)
print(lowyear_row.sum())
ns_book5.loc[lowyear_row, '발행년도'] = -1

print(ns_book5.eq(-1).sum()) 

6
번호          0
도서명         0
저자          0
출판사         0
발행년도       86
ISBN        0
세트 ISBN     0
부가기호        0
권           0
주제분류번호      0
도서권수        0
대출건수        0
등록일자        0
dtype: int64


In [6]:
#제목, 저자, 출판사, 발행년도 누락 확인

title_na = ns_book5['도서명'].isna() | ns_book5['도서명'].astype(str).str.strip().eq('')
author_na = ns_book5['저자'].isna() | ns_book5['저자'].str.strip().eq('')
pub_na = ns_book5['출판사'].isna() | ns_book5['출판사'].str.strip().eq('')
year_na = ns_book5['발행년도'].eq(-1)

na_row = title_na | author_na | pub_na | year_na

ns_book5['ISBN'].head(20)


0     9788937444319
1     9791190123969
2     9788968332982
3     9788970759906
4     9788934990833
5     9791189550370
6     9791160075793
7     9788960906679
8     9791158392512
9     9788998690557
10    9791190978088
11    9788901249599
12    9791196756987
13    9788972917335
14    9791185415420
15    9791189584887
16    9791189346201
17    9791188941605
18    9791188167432
19    9791170431725
Name: ISBN, dtype: object

In [16]:
import requests
import re
import pandas as pd
from bs4 import BeautifulSoup
import numpy as np

def get_bookinfo(row):
    title = row['도서명']
    author = row['저자']
    pub = row['출판사']
    year = row['발행년도']
    
    headers = {"User-Agent":"Mozilla/5.0"}
    url = r"https://www.yes24.com/product/search?domain=ALL&query={}"
    
    #html 텍스트 가져오기
    try:
        r = requests.get(url.format(row['ISBN']), headers=headers, timeout=3)
        r.raise_for_status()
    except requests.RequestException:
        return title, author, pub, year
    
    #html 수프 만들기
    soup = BeautifulSoup(r.text, 'html.parser')
    
    #전체 박스 만들기
    box = soup.select_one('div.item_info')
    if box is None:
        return title, author, pub, year
    
    #도서명 텍스트 찾기
    if pd.isna(title):
        title_a = box.select_one('a.gd_name')
        title = title_a.get_text(" ", strip=True) if title_a else None
    
    #저자 텍스트 찾기
    if pd.isna(author):
        a_tag= box.select_one('span.info_auth')
        a_list = a_tag.select('a') if a_tag else []
        author_list = [a.get_text(strip=True) for a in a_list]
        author = ', '.join(author_list) if author_list else None
        
    #출판사 텍스트 찾기
    if pd.isna(pub):
        pub_span_a = box.select_one('span.info_pub a')
        pub = pub_span_a.get_text(' ', strip=True) if pub_span_a else None
        
    #연도 텍스트 찾기
    if year == -1:
        date_tag = box.select_one('span.info_date')
        text = date_tag.get_text(strip=True) if date_tag else None

        m = re.search(r'^(\d{4})', text) if text else None
        year = m.group(1) if m else None
        
    return title, author, pub, year



In [ ]:
import gdown
import pandas as pd
from bs4 import BeautifulSoup

gdown.download(r'https://bit.ly/3GisL6J', 'ns_book4.csv', quiet=False)
ns_book = pd.read_csv('ns_book4.csv', encoding='utf-8', low_memory=False)

def data_fixing(ns_book):
    #도서권수, 대출건수 int 64로 바꾸기
    ns_book = ns_book.astype({'도서권수':'int64', '대출건수':'int64'})
    
    #NaN인 세트 ISBN 공백으로
    ns_book.fillna({'세트 ISBN':''}, inplace=True)
    
    setISBN_row = ns_book['세트 ISBN'].isna()
    ns_book.loc[setISBN_row, '세트 ISBN'] = '' 
    
    #발행년도 4자리로 찾기, 나머지는 -1로 바꾸기
    ns_book = ns_book.replace({'발행년도':{r'.*(\d{4}).*': r'\1'}}, regex=True)
    year_str_row = ns_book['발행년도'].astype(str).str.contains(r'\D', na=True)
    ns_book.loc[year_str_row, '발행년도'] = '-1'
    
    ns_book['발행년도'] = ns_book['발행년도'].astype(str).str.strip()
    ns_book.loc[ns_book['발행년도'].eq(''), '발행년도'] = '-1'
    
    ns_book = ns_book.astype({'발행년도':'int32'})
    
    dk_years_row = ns_book['발행년도'].gt(4000)
    ns_book.loc[dk_years_row, '발행년도'] = ns_book.loc[dk_years_row, '발행년도'] - 2333
    ns_book.loc[ns_book['발행년도'].gt(4000), '발행년도'] = -1
    
    old_years_row = ns_book['발행년도'].gt(0) & ns_book['발행년도'].lt(1900)
    ns_book.loc[old_years_row, '발행년도'] = -1
   
    #문제되는 데이터 로우 찾기
    NA_row = ns_book['도서명'].isna() | ns_book['저자'].isna() \
        | ns_book['출판사'].isna() | ns_book['발행년도'].eq(-1)
    
    updated_data = ns_book[NA_row].apply(get_bookinfo, axis=1, result_type = 'expand')
    updated_data.columns = ['도서명', '저자', '출판사', '발행년도']
    
    
    #데이터 누락된 정보 채우기
    ns_book.update(updated_data)
    
    #남은 결과 삭제하기
    ns_book_update = ns_book.dropna(subset=['도서명', '저자', '출판사'])
    ns_book_update = ns_book_update[ns_book_update['발행년도'] != -1]
    
    return ns_book_update


Downloading...
From: https://bit.ly/3GisL6J
To: c:\data\ns_book4.csv
100%|██████████| 55.5M/55.5M [00:10<00:00, 5.31MB/s]


In [ ]:
data_fixing(ns_book)

C:\Users\박중현\AppData\Local\Temp\ipykernel_8288\1709050190.py:44: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[2021 2021 2021 ... 1996 1996 1995]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  ns_book.update(updated_data)


,번호,도서명,저자,출판사,발행년도,ISBN,세트 ISBN,부가기호,권,주제분류번호,도서권수,대출건수,등록일자
0,1,인공지능과 흙,김동훈 지음,민음사,2021,9788937444319,,NaN,NaN,NaN,1,0,2021-03-19
1,2,가짜 행복 권하는 사회,김태형 지음,갈매나무,2021,9791190123969,,NaN,NaN,NaN,1,0,2021-03-19
2,3,나도 한 문장 잘 쓰면 바랄 게 없겠네,김선영 지음,블랙피쉬,2021,9788968332982,,NaN,NaN,NaN,1,0,2021-03-19
3,4,예루살렘 해변,"이도 게펜 지음, 임재희 옮김",문학세계사,2021,9788970759906,,NaN,NaN,NaN,1,0,2021-03-19
4,5,김성곤의 중국한시기행 : 장강·황하 편,김성곤 지음,김영사,2021,9788934990833,,NaN,NaN,NaN,1,0,2021-03-19
...,...,...,...,...,...,...,...,...,...,...,...,...,...
384586,401674,소설의 사회사 비교론,조동일 지음,지식산업사,2001,9788942340262,9788942300365,NaN,3,809.3,1,0,1970-01-01
384587,401677,큰오빠,박정근 지음,우리문학사,1998,9788974231323,9788974231309,NaN,2,813.6,2,0,1970-01-01
384588,401678,韓國現代詩大系,채만묵 編著,한국문화사,1996,9788977352971,9788977352988,NaN,3,811.608,1,0,1970-01-01
384589,401679,뉴 웨이브,제임스 모나코 지음,한나래,1996,9788985367448,9788985367424,NaN,2,688.04,1,0,1970-01-01


In [20]:
fixed_book = data_fixing(ns_book)

C:\Users\박중현\AppData\Local\Temp\ipykernel_8288\1709050190.py:44: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[2021 2021 2021 ... 1996 1996 1995]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  ns_book.update(updated_data)


In [23]:
fixed_book.isna().sum()
fixed_book.to_csv('my_ns_book6.csv', index=False, encoding='utf-8')
